## Consuming data using Kafka and Visualise

## Part 3: Kafka Consumer and Real-Time Visualisation

This section consumes the streaming outputs published from Task 2.8 using Kafka consumers. Spark is not used in this part. The Kafka consumers read prediction results from the Kafka topics created in Part 2 and update three real-time visualisations.

The dashboard contains three plots: a plot showing the number of high-severity accidents over time, a cumulative distribution of accidents by predicted severity, and a bubble map showing the location of high-severity accidents. The dashboard is updated repeatedly as new Kafka messages are received.

In [9]:
!pip install -q folium

In [10]:
# Part 3: Kafka consumer and visualisation setup

from kafka import KafkaConsumer
from json import loads
from collections import defaultdict
from IPython.display import clear_output

import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import os
import json
import math
import folium
from urllib.parse import quote

from IPython.display import IFrame, display, clear_output
from ipywidgets import Output


# Kafka configuration
hostip = "kafka"

high_severity_topic = "a2b_high_severity"
severity_count_topic = "a2b_severity_count"
district_severity_topic = "a2b_district_severity"


def connect_kafka_consumer(topic_names, offset="latest"):
    """
    Create a Kafka consumer for one or more Kafka topics.
    """
    consumer = None

    try:
        if isinstance(topic_names, str):
            topic_names = [topic_names]

        consumer = KafkaConsumer(
            *topic_names,
            bootstrap_servers=[f"{hostip}:9092"],
            auto_offset_reset=offset,
            enable_auto_commit=True,
            value_deserializer=lambda x: loads(x.decode("utf-8")),
            consumer_timeout_ms=1000,
            api_version=(3, 9)
        )

        print("Kafka consumer connected to topic(s):", topic_names)

    except Exception as ex:
        print("Exception while connecting Kafka consumer.")
        print(str(ex))

    finally:
        return consumer

### Real-time Visualisation

In [11]:
# Plot 1: High-severity accidents over time
def plot_high_severity_over_time(ax, high_severity_records):
    """
    Plot the number of high-severity accidents over time.
    """
    if len(high_severity_records) > 0:
        high_df = pd.DataFrame(high_severity_records)

        high_df["accident_ts_time"] = pd.to_datetime(
            high_df["accident_ts_time"],
            errors="coerce"
        )

        high_df = high_df.dropna(subset=["accident_ts_time"])
        high_df["time_window"] = high_df["accident_ts_time"].dt.floor("10s")

        high_count_by_time = (
            high_df
            .groupby("time_window")
            .size()
            .reset_index(name="count")
            .sort_values("time_window")
            .tail(30)
        )
        
        # Create continuous 10-second time windows so missing periods appear as 0.
        if len(high_count_by_time) > 0:
            full_time_range = pd.date_range(
                start=high_count_by_time["time_window"].min(),
                end=high_count_by_time["time_window"].max(),
                freq="10s"
            )

            high_count_by_time = (
                high_count_by_time
                .set_index("time_window")
                .reindex(full_time_range, fill_value=0)
                .rename_axis("time_window")
                .reset_index()
                .tail(30)
            )

        ax.plot(
            high_count_by_time["time_window"],
            high_count_by_time["count"],
            marker="o"
        )

        ax.fill_between(
            high_count_by_time["time_window"],
            high_count_by_time["count"],
            alpha=0.2
        )
        
        # Add count labels above each point.
        for x_value, y_value in zip(
            high_count_by_time["time_window"],
            high_count_by_time["count"]
        ):
            ax.annotate(
                str(y_value),
                xy=(x_value, y_value),
                xytext=(0, 8),
                textcoords="offset points",
                ha="center",
                fontsize=12
            )

        ax.set_title("High-Severity Accidents Over Time (Severity > 7)")
        ax.set_xlabel("Accident Time Window")
        ax.set_ylabel("Number of High-Severity Accidents")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.tick_params(axis="x", rotation=45)

    else:
        ax.set_title("High-Severity Accidents Over Time")
        ax.text(0.5, 0.5, "Waiting for data", ha="center", va="center")

In [12]:
# Plot 2: Cumulative distribution by predicted severity

def get_severity_bar_color(severity):
    """
    Return a professional risk-gradient colour
    based on predicted severity level.
    """
    severity_colours = {
        1: "#2E7D32",   # green
        2: "#43A047",   # light green
        3: "#00897B",   # teal
        4: "#1976D2",   # blue
        5: "#FBC02D",   # yellow
        6: "#F9A825",   # amber
        7: "#EF6C00",   # orange
        8: "#E64A19",   # deep orange
        9: "#C62828",   # red
        10: "#7F0000"   # dark red
    }

    return severity_colours.get(severity, "#757575")


def plot_severity_distribution(ax, severity_window_counts):
    """
    Plot the cumulative distribution of accidents by predicted severity.
    """
    severity_totals = defaultdict(int)

    for key, count in severity_window_counts.items():
        predicted_severity = key[2]
        severity_totals[predicted_severity] += count

    if len(severity_totals) > 0:
        severity_labels = list(range(1, 11))

        severity_values = [
            severity_totals[label]
            for label in severity_labels
        ]

        severity_colours = [
            get_severity_bar_color(label)
            for label in severity_labels
        ]

        bars = ax.bar(
            severity_labels,
            severity_values,
            color=severity_colours,
            edgecolor="white",
            linewidth=1.2
        )

        ax.bar_label(
            bars,
            labels=[f"{value:,}" for value in severity_values],
            padding=3,
            fontsize=12
        )

        ax.set_title("Cumulative Accidents by Predicted Severity")
        ax.set_xlabel("Predicted Severity")
        ax.set_ylabel("Total Accidents")
        ax.set_xticks(range(1, 11))
        ax.margins(y=0.15)

        ax.grid(
            axis="y",
            linestyle="--",
            alpha=0.25
        )

    else:
        ax.set_title("Cumulative Accidents by Predicted Severity")
        ax.text(0.5, 0.5, "Waiting for data", ha="center", va="center")

In [13]:
# Plot 3: Simple live Folium bubble map
# The base map is created once.
# Bubble data is updated through a JavaScript file.

import os
import json
import folium
import pandas as pd

from urllib.parse import quote
from branca.element import MacroElement, Template


def get_jupyter_file_src(file_path, jupyter_root="/home/student"):
    """
    Convert a local file path into a Jupyter /files/ URL.
    """
    relative_path = os.path.relpath(
        os.path.abspath(file_path),
        jupyter_root
    )

    return "/files/" + quote(relative_path)


def build_bubble_payload(high_severity_records, max_groups=150):
    """
    Prepare bubble data for the map.

    Each bubble represents a nearby location cluster.
    Bubble size represents total high-severity accidents in that cluster.
    """
    if len(high_severity_records) == 0:
        return {"records": []}

    map_df = pd.DataFrame(high_severity_records).copy()

    required_columns = ["longitude", "latitude", "predicted_severity"]

    for column in required_columns:
        if column not in map_df.columns:
            return {"records": []}

    map_df["longitude"] = pd.to_numeric(
        map_df["longitude"],
        errors="coerce"
    )

    map_df["latitude"] = pd.to_numeric(
        map_df["latitude"],
        errors="coerce"
    )

    map_df["predicted_severity"] = pd.to_numeric(
        map_df["predicted_severity"],
        errors="coerce"
    )

    map_df = map_df.dropna(
        subset=["longitude", "latitude", "predicted_severity"]
    )

    # Keep plausible UK coordinate range
    map_df = map_df[
        (map_df["longitude"].between(-10, 3)) &
        (map_df["latitude"].between(49, 62))
    ]

    if len(map_df) == 0:
        return {"records": []}

    # Remove duplicate Kafka records if they exist
    if "collision_index" in map_df.columns and "accident_ts_time" in map_df.columns:
        map_df = map_df.drop_duplicates(
            subset=["collision_index", "accident_ts_time"],
            keep="last"
        )

    # Group nearby points to reduce overlapping bubbles.
    map_df["lat_group"] = map_df["latitude"].round(1)
    map_df["lon_group"] = map_df["longitude"].round(1)

    group_columns = ["lat_group", "lon_group"]

    if "local_district" in map_df.columns:
        group_columns.append("local_district")

    bubble_df = (
        map_df
        .groupby(group_columns, dropna=False)
        .agg(
            total_accidents=("predicted_severity", "count"),
            max_severity=("predicted_severity", "max"),
            avg_severity=("predicted_severity", "mean"),
            latitude=("latitude", "mean"),
            longitude=("longitude", "mean")
        )
        .reset_index()
    )

    if "accident_ts_time" in map_df.columns:
        latest_time_df = (
            map_df
            .groupby(group_columns, dropna=False)
            .agg(latest_accident_time=("accident_ts_time", "max"))
            .reset_index()
        )

        bubble_df = bubble_df.merge(
            latest_time_df,
            on=group_columns,
            how="left"
        )
    else:
        bubble_df["latest_accident_time"] = "N/A"

    if "local_district" not in bubble_df.columns:
        bubble_df["local_district"] = "N/A"

    bubble_df["local_district"] = bubble_df["local_district"].fillna("N/A")
    bubble_df["latest_accident_time"] = bubble_df["latest_accident_time"].fillna("N/A")

    # Keep only the largest clusters to keep the map readable
    bubble_df = (
        bubble_df
        .sort_values(
            ["total_accidents", "max_severity"],
            ascending=False
        )
        .head(max_groups)
    )

    records = []

    for _, row in bubble_df.iterrows():
        records.append({
            "latitude": float(row["latitude"]),
            "longitude": float(row["longitude"]),
            "total_accidents": int(row["total_accidents"]),
            "max_severity": float(row["max_severity"]),
            "avg_severity": float(row["avg_severity"]),
            "local_district": str(row["local_district"]),
            "latest_accident_time": str(row["latest_accident_time"])
        })

    return {"records": records}


def write_bubble_js(high_severity_records, js_path):
    """
    Write bubble data into a JavaScript file.
    The Folium map loads this file repeatedly without reloading the base map.
    """
    payload = build_bubble_payload(high_severity_records)

    js_text = f"""
window.a2bBubbleData = {json.dumps(payload, allow_nan=False)};

if (window.updateA2BBubblesFromData) {{
    window.updateA2BBubblesFromData(window.a2bBubbleData);
}}
"""

    temporary_path = js_path + ".tmp"

    with open(temporary_path, "w") as file:
        file.write(js_text)

    os.replace(temporary_path, js_path)


class SimpleLiveBubbleLayer(MacroElement):
    """
    Simple Folium layer that updates bubbles from a JavaScript data file.
    """
    _template = Template("""
    {% macro script(this, kwargs) %}

        var mapObject = {{ this._parent.get_name() }};
        var bubbleLayer = L.layerGroup().addTo(mapObject);
        var hasFitBounds = false;

        var statusControl = L.control({position: "topright"});

        statusControl.onAdd = function(map) {
            var div = L.DomUtil.create("div", "bubble-status-control");
            div.style.backgroundColor = "white";
            div.style.padding = "8px";
            div.style.border = "2px solid grey";
            div.style.borderRadius = "6px";
            div.style.fontSize = "12px";
            div.innerHTML = "Waiting for bubble data...";
            return div;
        };

        statusControl.addTo(mapObject);

        var labelStyle = document.createElement("style");
        labelStyle.innerHTML = `
            .bubble-count-label {
                background: transparent;
                border: none;
                box-shadow: none;
                color: white;
                font-size: 10px;
                text-align: center;
            }

            .bubble-count-label::before {
                display: none;
            }
        `;
        document.head.appendChild(labelStyle);

        function updateStatus(message) {
            var statusBox = document.querySelector(".bubble-status-control");

            if (statusBox) {
                statusBox.innerHTML = message;
            }
        }

        function getSeverityColor(severity) {
            if (severity >= 10) {
                return "darkred";
            } else if (severity >= 9) {
                return "red";
            } else if (severity >= 8) {
                return "orange";
            } else {
                return "blue";
            }
        }

        function escapeHtml(value) {
            return String(value)
                .replace(/&/g, "&amp;")
                .replace(/</g, "&lt;")
                .replace(/>/g, "&gt;")
                .replace(/"/g, "&quot;")
                .replace(/'/g, "&#039;");
        }

        function getBubbleRadius(totalAccidents) {
            // Larger pixel radius so bubbles remain visible when zooming.
            var radius = 6 + (Math.sqrt(totalAccidents) * 4.0);

            radius = Math.max(radius, 6);
            radius = Math.min(radius, 35);

            return radius;
        }

        window.updateA2BBubblesFromData = function(data) {

            bubbleLayer.clearLayers();

            if (!data || !data.records || data.records.length === 0) {
                updateStatus("Bubble clusters: 0");
                mapObject.invalidateSize();
                return;
            }

            var bounds = [];

            data.records.forEach(function(row) {

                var latitude = Number(row.latitude);
                var longitude = Number(row.longitude);
                var totalAccidents = Number(row.total_accidents);
                var maxSeverity = Number(row.max_severity);
                var avgSeverity = Number(row.avg_severity);

                if (isNaN(latitude) || isNaN(longitude)) {
                    return;
                }

                var colour = getSeverityColor(maxSeverity);
                var radius = getBubbleRadius(totalAccidents);

                var popupHtml = `
                    <b>Local District:</b> ${escapeHtml(row.local_district)}<br>
                    <b>Total High-Severity Accidents:</b> ${totalAccidents}<br>
                    <b>Max Severity:</b> ${maxSeverity.toFixed(0)}<br>
                    <b>Average Severity:</b> ${avgSeverity.toFixed(2)}<br>
                    <b>Latest Accident Time:</b> ${escapeHtml(row.latest_accident_time)}<br>
                    <b>Latitude:</b> ${latitude.toFixed(5)}<br>
                    <b>Longitude:</b> ${longitude.toFixed(5)}
                `;

                var marker = L.circleMarker([latitude, longitude], {
                radius: radius,
                stroke: true,
                color: "white",
                weight: 1,
                opacity: 0.9,
                fillColor: colour,
                fillOpacity: 0.65
            });

                marker.bindPopup(popupHtml);

                if (totalAccidents > 1) {
                    marker.bindTooltip(
                        String(totalAccidents),
                        {
                            permanent: true,
                            direction: "center",
                            className: "bubble-count-label"
                        }
                    );
                }

                marker.addTo(bubbleLayer);
                bounds.push([latitude, longitude]);
            });

            updateStatus("Bubble clusters: " + data.records.length);

            if (!hasFitBounds && bounds.length > 0) {
                mapObject.fitBounds(bounds, {
                    padding: [20, 20]
                });

                hasFitBounds = true;
            }

            mapObject.invalidateSize();
        };

        function loadBubbleScript() {
            var oldScript = document.getElementById("bubble-data-script");

            if (oldScript) {
                oldScript.remove();
            }

            var script = document.createElement("script");
            script.id = "bubble-data-script";
            script.src = "{{ this.js_url }}?t=" + new Date().getTime();

            script.onerror = function() {
                updateStatus("Bubble JS load error");
            };

            document.body.appendChild(script);
        }

        loadBubbleScript();
        setInterval(loadBubbleScript, 5000);

    {% endmacro %}
    """)

    def __init__(self, js_url):
        super().__init__()
        self._name = "SimpleLiveBubbleLayer"
        self.js_url = js_url


def create_live_folium_map_shell(
    map_html_path="a2b_live_folium_map.html",
    js_filename="a2b_bubble_data.js"
):
    """
    Create the Folium base map once.
    """
    write_bubble_js([], js_filename)

    js_url = get_jupyter_file_src(js_filename)

    accident_map = folium.Map(
        location=[54.5, -2.5],
        zoom_start=6,
        tiles="OpenStreetMap",
        width="100%",
        height="650px"
    )

    legend_html = """
    <div style="
        position: fixed;
        bottom: 40px;
        left: 40px;
        width: 245px;
        z-index: 9999;
        background-color: white;
        border: 2px solid grey;
        border-radius: 6px;
        padding: 10px;
        font-size: 13px;
    ">
    <b>Predicted Severity</b><br>
    <span style="color: orange;">●</span> Severity 8<br>
    <span style="color: red;">●</span> Severity 9<br>
    <span style="color: darkred;">●</span> Severity 10<br><br>
    Bubble size = total accidents
    </div>
    """

    accident_map.get_root().html.add_child(
        folium.Element(legend_html)
    )

    accident_map.add_child(
        SimpleLiveBubbleLayer(js_url=js_url)
    )

    accident_map.save(map_html_path)

In [14]:
# Styling cell
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 18,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

In [15]:
# Real-time dashboard using Kafka consumer output

# 1. Connect Kafka consumer
dashboard_consumer = connect_kafka_consumer(
    [
        high_severity_topic,
        severity_count_topic,
        district_severity_topic
    ],
    offset="latest"
)

# 2. Initialise dashboard storage
high_severity_records = []
severity_window_counts = {}
district_window_counts = {}

dashboard_duration_seconds = 600
refresh_interval_seconds = 5
recent_map_record_limit = 1200

# 3. Create live Folium map once
map_html_path = f"a2b_live_folium_map_{int(time.time())}.html"
bubble_js_path = "a2b_bubble_data.js"

create_live_folium_map_shell(
    map_html_path=map_html_path,
    js_filename=bubble_js_path
)

map_iframe_src = get_jupyter_file_src(map_html_path) + f"?v={int(time.time())}"

plots_output = Output()

display(plots_output)

display(
    IFrame(
        src=map_iframe_src,
        width="100%",
        height=680
    )
)

# 4. Start dashboard loop
start_time = time.time()

try:
    while time.time() - start_time < dashboard_duration_seconds:

        # 5. Read Kafka messages
        messages = dashboard_consumer.poll(timeout_ms=1000)

        for topic_partition, records in messages.items():

            for message in records:
                topic = message.topic
                data = message.value

                if topic == high_severity_topic:
                    high_severity_records.append(data)

                elif topic == severity_count_topic:
                    key = (
                        data.get("window_start"),
                        data.get("window_end"),
                        int(data.get("predicted_severity", 0))
                    )

                    severity_window_counts[key] = int(
                        data.get("total_accidents", 0)
                    )

                elif topic == district_severity_topic:
                    key = (
                        data.get("window_start"),
                        data.get("window_end"),
                        data.get("local_district")
                    )

                    district_window_counts[key] = data

        # 6. Keep only recent high-severity records for the map
        if len(high_severity_records) > recent_map_record_limit:
            high_severity_records = high_severity_records[-recent_map_record_limit:]

        # 7. Update live bubble map data
        write_bubble_js(
            high_severity_records,
            bubble_js_path
        )

        bubble_payload = build_bubble_payload(high_severity_records)
        bubble_cluster_count = len(bubble_payload.get("records", []))

        # 8. Refresh Plot 1 and Plot 2 only
        with plots_output:
            clear_output(wait=True)

            fig, axes = plt.subplots(
                2, 1,
                figsize=(16, 12)
            )

            # Plot 1
            plot_high_severity_over_time(
                axes[0],
                high_severity_records
            )

            # Plot 2
            plot_severity_distribution(
                axes[1],
                severity_window_counts
            )

            plt.tight_layout(h_pad=4)
            plt.show()
            plt.close(fig)

            # demo status
            print("Dashboard running")
            print("High-severity records used for Plot 1 and map:", len(high_severity_records))
            print("Live bubble clusters on map:", bubble_cluster_count)
            print("Severity summary windows used for Plot 2:", len(severity_window_counts))
            print("District summary records received:", len(district_window_counts))
            print("Map status: Folium base map is fixed; bubbles update every 5 seconds.")

        time.sleep(refresh_interval_seconds)

except KeyboardInterrupt:
    print("Dashboard stopped by user.")

finally:
    dashboard_consumer.close()
    print("Dashboard consumer closed.")

Kafka consumer connected to topic(s): ['a2b_high_severity', 'a2b_severity_count', 'a2b_district_severity']


Output()

Dashboard consumer closed.


### Real-Time Kafka Dashboard

This dashboard consumes streaming messages from the Kafka topics published in Task 2.8. Spark is not used in this part. The dashboard uses a Kafka consumer to read high-severity accident records from `a2b_high_severity` and severity count summaries from `a2b_severity_count`.

Three visualisations are updated repeatedly. The first plot shows the number of high-severity accidents over time using 10-second accident time windows. The second plot shows the cumulative distribution of accidents by predicted severity level. The third plot shows a bubble map of high-severity accident locations using longitude and latitude, with the bubble size based on the predicted severity.

The consumer uses `offset="latest"` so the dashboard behaves like a live monitoring dashboard. It only consumes new Kafka messages produced after the dashboard starts, instead of replaying older messages already stored in the Kafka topic.

## Generative AI Usage Statement

Generative AI was used selectively in this assignment as a learning and support tool. The final code, testing decisions, interpretation of outputs, and notebook organisation were reviewed, adapted, and executed by me. I used Generative AI mainly to clarify concepts, improve code readability, debug errors, and draft explanatory markdown. I understand that I am responsible for the correctness, quality, and academic integrity of the submitted work.

### Task 3: Kafka Consumer and Real-Time Visualisation

For Task 3, Generative AI was used to support the design and refinement of the Kafka consumer dashboard. It helped explain how to consume multiple Kafka topics, organise the visualisation code into separate plotting functions, and improve the readability of the dashboard. It also assisted in debugging layout issues, improving chart labels, adjusting the bubble map display, and explaining why different Kafka topics were used for different plots. The final dashboard implementation was tested using the Kafka topics produced in Task 2.8, and the visualisations were adjusted based on the actual output observed in my notebook.
